# I.i.d.-sampled lognormal kappa-spread sweep on the Colab T4

Sibling of `colab_ks_wealth_lognormal_sweep.ipynb`. Sweeps heterogeneous
capital productivity `kappa ~ LogNormal(mu, sigma)`, `mu = -sigma^2/2`
(mean fixed to 1), over `sigma` from 0 (homogeneous) to 1 (wide spread) --
same design as the sibling notebook, but kappas here are drawn as genuine
i.i.d. samples (`np.random.default_rng(seed)`) rather than placed on a
deterministic quantile mesh. Because a single draw no longer pins down what
a given sigma looks like, this sweep also runs a list of seeds at every
sigma, so the spread of outcomes across independent draws is visible in
the results. Full design writeup: `runs/ks-wealth-lognormal-random/README.md`.

Default config: `n_agents=500`, `num_envs=8`,
`sigmas=[0.0,0.2,0.4,0.6,0.8,1.0]`, `seeds=[0,1,2,3,4]` (30 cells total).
**Time the first cell before assuming the rest fit in one Colab session.**
results.csv and every cell's figures/raw rollout are checkpointed as each
cell finishes, so a disconnect partway through only loses the run in
progress.

In [ ]:
# Setup: clone or update the repo, install (idempotent -- safe to re-run).
%cd /content
![ -d jax-marl-bc ] || git clone https://github.com/danmonuni/jax-marl-bc.git
%cd jax-marl-bc
!git pull
!pip install -q -r requirements.txt && pip install -q -e . --no-deps

In [ ]:
# Sanity: a GPU runtime is attached (Runtime > Change runtime type > T4 GPU).
!nvidia-smi -L

In [ ]:
# Mount Drive BEFORE the run so results are saved as soon as they finish
# (a Colab disconnect then loses at most the run in progress, never a
# finished one).
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

def save_results(name='ks-wealth-lognormal-random'):
    """Sync runs/<name>/results -> Drive (exact path, idempotent re-sync)."""
    src = f'runs/{name}/results'
    dst = f'/content/drive/MyDrive/jax-marl-bc-runs/{name}/results'
    assert os.path.exists(src), f"{src} missing - did the sweep finish?"
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"saved {src} -> {dst}")

## Sigma x seed sweep

Runs `runs/ks-wealth-lognormal-random/config.yaml` as-is: `n_agents=500`,
`device=gpu`, `sigmas=[0.0,0.2,0.4,0.6,0.8,1.0]`, `seeds=[0,1,2,3,4]`. Pass
dotlist overrides after the script path to change any of these -- e.g. a
quick CPU smoke test with two sigmas and two seeds, or a wider seed list:
`!python runs/ks-wealth-lognormal-random/sweep_lognormal_random.py n_agents=50 device=cpu sim_steps=200 total_timesteps=2000 "sigmas=[0.0,0.5]" "seeds=[0,1]"`
`!python runs/ks-wealth-lognormal-random/sweep_lognormal_random.py "seeds=[0,1,2,3,4,5,6,7,8,9]"`

In [ ]:
!python runs/ks-wealth-lognormal-random/sweep_lognormal_random.py

In [ ]:
save_results('ks-wealth-lognormal-random')

## Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv('runs/ks-wealth-lognormal-random/results/results.csv')
display(df[['sigma', 'seed', 'kappa_std', 'capital_gini', 'top_0.1_share', 'top_0.01_share', 'K_mean', 'euler_mean_abs']])

display(Image('runs/ks-wealth-lognormal-random/results/comparison.png'))

## Per-cell steady-state dashboards

One dashboard per `(sigma, seed)` cell: kappa profile, aggregate
capital/consumption paths, the aggregate KS shock, the Lorenz curve, and the
wealth histogram.

In [ ]:
for _, row in df.iterrows():
    sigma, seed = row['sigma'], int(row['seed'])
    print(f"sigma = {sigma:.2f}  seed = {seed}")
    display(Image(f'runs/ks-wealth-lognormal-random/results/figures/sigma_{sigma:.2f}_seed_{seed}_steady_state.png'))